# EduFeed DeBERTa Pipeline — Kaggle T4 x2 (4 TPU)

**Dataset:** `balanced_edufeed_dataset.xlsx` — 35,421 rows, balanced 3-class sentiment  
**Model:** `cross-encoder/nli-deberta-v3-base` (zero-shot sentiment) + `all-MiniLM-L6-v2` (SBERT embeddings)  
**Accelerator:** GPU T4 x2  

### Steps
1. Install dependencies  
2. Verify GPU  
3. Upload dataset (as Kaggle Dataset)  
4. Ingest & clean  
5. Anonymise professors  
6. SBERT embeddings  
7. DeBERTa zero-shot sentiment (with ground-truth labels in this dataset)  
8. FAISS index  
9. Insight cards  
10. Evaluation  
11. Export outputs

## Step 0 — Kaggle Setup Note

> **Before running:**  
> 1. Upload `balanced_edufeed_dataset.xlsx` as a Kaggle Dataset (New Dataset → upload file).  
> 2. In your notebook: **Add data** → find your dataset → add it.  
> 3. In **Settings** → **Accelerator** → select **GPU T4 x2**.  
> 4. The dataset will appear at `/kaggle/input/<your-dataset-slug>/balanced_edufeed_dataset.xlsx`  
> 5. Update `DATASET_PATH` in Cell 3 to match.

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────
# Run once. Restart kernel NOT needed — packages are importable immediately.
import subprocess, sys

pkgs = [
    "sentence-transformers",
    "transformers>=4.40.0",
    "faiss-gpu",          # GPU-accelerated FAISS (T4 supported on Kaggle)
    "accelerate>=0.26.0",
    "datasets",
    "openpyxl",
    "scikit-learn",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs,
    capture_output=True, text=True
)
# Show only errors (if any)
if result.returncode != 0:
    print(result.stderr[-2000:])
else:
    print("All packages installed ✔")

In [ ]:
# ── Cell 2: Verify GPU ───────────────────────────────────────────────
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}:", torch.cuda.get_device_name(i),
              f"| VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")
    # Quick sanity check
    a = torch.randn(512, 512).cuda()
    b = torch.randn(512, 512).cuda()
    print("GPU matmul test:", torch.mm(a, b).shape, "✔")
else:
    print("WARNING: No GPU found. Check Accelerator setting in notebook Settings.")

In [ ]:
# ── Cell 3: Config — UPDATE DATASET_PATH to match your Kaggle dataset slug ──
import os
from pathlib import Path

# *** Update this path to match your uploaded dataset slug ***
# Example: /kaggle/input/edufeed-balanced/balanced_edufeed_dataset.xlsx
DATASET_PATH = "/kaggle/input/balanced-edufeed-dataset/balanced_edufeed_dataset.xlsx"

ROOT = Path("/kaggle/working")
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

# DeBERTa batch size — T4 x2 with 16 GB VRAM each; 64 is safe
DEBERTA_BATCH = 64
SBERT_BATCH   = 256   # MiniLM is small, can use larger batch
MAX_PROF_CARDS = None  # None = all professors

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Dataset path:", DATASET_PATH)
print("File exists:", os.path.exists(DATASET_PATH))

In [ ]:
# ── Cell 4: Ingest & Clean ───────────────────────────────────────────
import pandas as pd
import numpy as np

df = pd.read_excel(DATASET_PATH, engine="openpyxl")
print(f"Raw shape: {df.shape}")
print("Columns:", df.columns.tolist())

# Drop rows missing critical fields
df = df.dropna(subset=["professor_name", "comments", "star_rating"])
df = df.drop_duplicates(subset=["professor_name", "comments"])
df = df.reset_index(drop=True)

# Normalise text
df["comments"] = df["comments"].fillna("").astype(str).str.strip()
df["star_rating"] = pd.to_numeric(df["star_rating"], errors="coerce")

# Normalise sentiment_label (Positive → positive)
df["sentiment_label"] = df["sentiment_label"].astype(str).str.lower().str.strip()
valid_sentiments = {"positive", "negative", "neutral"}
df = df[df["sentiment_label"].isin(valid_sentiments)]
df = df.reset_index(drop=True)

# Add review ID
df.insert(0, "review_id", df.index.astype(str).str.zfill(6))

print(f"\nClean shape: {df.shape}")
print("Sentiment distribution:", df["sentiment_label"].value_counts().to_dict())
print("Professors:", df["professor_name"].nunique())
print("Sample:")
display(df[["review_id", "professor_name", "star_rating", "sentiment_label", "comments"]].head(3))

In [ ]:
# ── Cell 5: Anonymise Professors ────────────────────────────────────
import hashlib

def sha256_token(name: str) -> str:
    """Deterministic, one-way professor token."""
    return "PROF_" + hashlib.sha256(name.lower().strip().encode()).hexdigest()[:8].upper()

prof_map = {n: sha256_token(n) for n in df["professor_name"].unique()}
df["professor_token"] = df["professor_name"].map(prof_map)

# Remove raw name (privacy)
df = df.drop(columns=["professor_name"])

PROF_COL = "professor_token"
print(f"Anonymised {len(prof_map)} professors ✔")
print("Sample tokens:", list(prof_map.values())[:5])

In [ ]:
# ── Cell 6: SBERT Embeddings (GPU) ──────────────────────────────────
from sentence_transformers import SentenceTransformer

print("Loading SentenceTransformer...")
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

texts = df["comments"].tolist()
print(f"Encoding {len(texts):,} texts on {DEVICE}...")

emb = sbert.encode(
    texts,
    batch_size=SBERT_BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

np.save(ROOT / "layer3_embeddings.npy", emb)
print(f"Embeddings shape: {emb.shape} ✔")

# Free SBERT from GPU memory
del sbert
torch.cuda.empty_cache()

In [ ]:
# ── Cell 7: DeBERTa Zero-Shot Sentiment ─────────────────────────────
# Uses cross-encoder/nli-deberta-v3-base via zero-shot-classification
# T4 x2: device=0 uses first GPU. Multi-GPU inference isn't needed for this size.
from transformers import pipeline as hf_pipeline

SENTIMENT_LABELS = ["positive", "negative", "neutral"]

print("Loading DeBERTa zero-shot pipeline...")
clf = hf_pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-base",
    device=0,                  # first T4
    batch_size=DEBERTA_BATCH,
    truncation=True,
    max_length=512,
)

print(f"Running DeBERTa on {len(texts):,} texts (batch={DEBERTA_BATCH})...")
print("Estimated time: ~12–18 min on T4 x1 for 35k rows")

predicted_sentiments = []
predicted_scores     = []

for i in range(0, len(texts), DEBERTA_BATCH):
    batch = texts[i : i + DEBERTA_BATCH]
    results = clf(batch, SENTIMENT_LABELS, multi_label=False)
    for r in results:
        predicted_sentiments.append(r["labels"][0])
        predicted_scores.append(round(r["scores"][0], 4))
    if i % 2000 == 0:
        pct = i / len(texts) * 100
        print(f"  {i:>6}/{len(texts):,}  ({pct:.1f}%)")

df["sentiment"]       = predicted_sentiments
df["sentiment_score"] = predicted_scores

df.to_csv(ROOT / "layer3_enriched.csv", index=False)
print("\nPredicted sentiment distribution:")
print(df["sentiment"].value_counts())

# Free GPU memory
del clf
torch.cuda.empty_cache()
print("DeBERTa inference done ✔")

In [ ]:
# ── Cell 8: FAISS Index ──────────────────────────────────────────────
import faiss

emb_f32 = emb.astype("float32")
dim = emb_f32.shape[1]

# HNSW index — fast approximate nearest-neighbour, no GPU required
index = faiss.IndexHNSWFlat(dim, 32)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 64
index.add(emb_f32)

faiss.write_index(index, str(ROOT / "layer4_faiss.index"))
print(f"FAISS index: {index.ntotal:,} vectors, dim={dim} ✔")

In [ ]:
# ── Cell 9: Professor Insight Cards ─────────────────────────────────
import json

professors = df[PROF_COL].unique()
if MAX_PROF_CARDS:
    professors = professors[:MAX_PROF_CARDS]

cards = []
for prof in professors:
    prof_df = df[df[PROF_COL] == prof]
    dept = (
        prof_df["department_name"].mode().iloc[0]
        if "department_name" in prof_df.columns and not prof_df["department_name"].isna().all()
        else "Unknown"
    )
    avg_rating = round(prof_df["star_rating"].mean(), 2) if "star_rating" in prof_df.columns else "N/A"
    sent_dist  = prof_df["sentiment"].value_counts().to_dict()
    top_reviews = prof_df.nlargest(3, "sentiment_score")["comments"].str[:200].tolist()

    cards.append({
        "professor":   prof,
        "department":  dept,
        "avg_rating":  avg_rating,
        "review_count": len(prof_df),
        "sentiments":  sent_dist,
        "top_reviews": top_reviews,
    })

insights_path = OUTPUTS / "insights.json"
with open(insights_path, "w") as f:
    json.dump(cards, f, indent=2)

print(f"Generated {len(cards)} insight cards ✔")
print("Sample card:")
print(json.dumps(cards[0], indent=2))

In [ ]:
# ── Cell 10: Evaluation ──────────────────────────────────────────────
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("=" * 60)
print("1. SENTIMENT ACCURACY (DeBERTa vs ground truth labels)")
print("=" * 60)

y_true = df["sentiment_label"]   # normalised in Cell 4
y_pred = df["sentiment"]

acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc*100:.2f}%\n")
print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

cm = confusion_matrix(y_true, y_pred, labels=["positive", "neutral", "negative"])
print("Confusion Matrix (positive / neutral / negative):")
print(cm)

# ── Retrieval Quality ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("2. RETRIEVAL QUALITY (FAISS Hit@1, Hit@5)")
print("=" * 60)

rng    = np.random.default_rng(42)
sample = min(500, len(df))
idx_sample = rng.choice(len(df), sample, replace=False)

hit1 = hit5 = 0
for i in idx_sample:
    q = emb_f32[i].reshape(1, -1)
    _, I = index.search(q, k=6)
    neighbors = [j for j in I[0] if j != i][:5]
    true_prof  = df.iloc[i][PROF_COL]
    nbr_profs  = [df.iloc[j][PROF_COL] for j in neighbors]
    if nbr_profs and nbr_profs[0] == true_prof: hit1 += 1
    if true_prof in nbr_profs:                   hit5 += 1

print(f"Hit@1 : {hit1/sample*100:.1f}%")
print(f"Hit@5 : {hit5/sample*100:.1f}%")

# ── Pipeline Summary ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. PIPELINE SUMMARY")
print("=" * 60)
print(f"Total reviews      : {len(df):,}")
print(f"Professors         : {df[PROF_COL].nunique():,}")
print(f"Insight cards      : {len(cards):,}")
print(f"Embedding dims     : {emb.shape[1]}")
print(f"FAISS index size   : {index.ntotal:,}")
print(f"Avg star rating    : {df['star_rating'].mean():.3f}")
print(f"Sentiment dist     : {df['sentiment'].value_counts().to_dict()}")
print(f"DeBERTa accuracy   : {acc*100:.2f}%")
print("=" * 60)

In [ ]:
# ── Cell 11: Export Outputs ──────────────────────────────────────────
from datetime import datetime
import shutil

# predictions CSV
df.to_csv(OUTPUTS / "predictions.csv", index=False)

# embeddings
shutil.copy2(ROOT / "layer3_embeddings.npy", OUTPUTS / "embeddings.npy")

# FAISS index
shutil.copy2(ROOT / "layer4_faiss.index",   OUTPUTS / "faiss.index")

# Summary report
report = f"""# EduFeed DeBERTa Pipeline Report
Generated : {datetime.now()}
Model     : cross-encoder/nli-deberta-v3-base (zero-shot)
Embeddings: sentence-transformers/all-MiniLM-L6-v2

## Data
- Rows       : {len(df):,}
- Professors : {df[PROF_COL].nunique():,}
- Avg rating : {df['star_rating'].mean():.3f}

## Results
- Sentiment accuracy : {acc*100:.2f}%
- Sentiment dist     : {df['sentiment'].value_counts().to_dict()}
- FAISS Hit@1        : {hit1/sample*100:.1f}%
- FAISS Hit@5        : {hit5/sample*100:.1f}%

## Output Files
- outputs/predictions.csv  — enriched dataset with predicted sentiments
- outputs/insights.json    — professor insight cards
- outputs/embeddings.npy   — SBERT embeddings (float32)
- outputs/faiss.index      — FAISS HNSW index
- outputs/report.md        — this report
Status: COMPLETE
"""

(OUTPUTS / "report.md").write_text(report)

print("Output files saved:")
for f in sorted(OUTPUTS.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<30} {size_mb:.2f} MB")

print("\n✅ PIPELINE COMPLETE")